# 02 - Data-generation ablation

Does physics-based scattering before forward modeling actually beat cheap post-hoc
decimation? Three training sets are built from one shared dense target (naive regular
decimation, naive scattered decimation, proposed physics-based scattering), a separate
U-Net is trained on each, and all three are cross-tested against all three test sets.

## 1. Build the three datasets and train three models

In [ ]:
# ==============================================================================
# Three-Way Data-Generation Ablation Study
#
# Consolidates naive regular decimation, naive scattered decimation,
# and physics-based scattering into a single cross-tested ablation script.
# ==============================================================================

import os
import time
import json
import numpy as np
import torch
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as ssim
from scipy.ndimage import gaussian_filter1d
import deepwave
from deepwave import scalar

# --- 1. Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)
print(f"Global seed set to {GLOBAL_SEED} ().")

# Limit TF memory to prevent conflicts
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f"TF Memory Warning: {e}")

# Ensure output directories exist
os.makedirs('outputs/datagen_ablation/figures', exist_ok=True)
os.makedirs('outputs/datagen_ablation/stacked', exist_ok=True)
os.makedirs('outputs/datagen_ablation/training_logs', exist_ok=True)

# --- 2. Parameters ---
dx = 2
dt = 0.004
freq = 25
nt = 300
peak_time = 1.5 / freq
nx, nz = 601, 501
d_receiver = nx / 300
source_depth_grid = int(60)
receiver_depth_grid = int(50)
first_source_grid = 5
first_receiver_grid = 0

N_SHOTS_GOOD = 100
N_SHOTS_BAD = 15
MAX_OFFSET_M = 150.0

WINDOW_SIZE = 128
STEP_SIZE = 5
RANDOM_STATE = 42
TEST_SIZE_TEMP = 0.3
VALIDATION_SIZE_FINAL = 0.3
BATCH_SIZE = 32

print("Parameters set, matching verified training pipeline exactly.")

# --- 3. Helper Functions ---
def normalize_data(data, method='minmax_symmetric'):
    if method == 'minmax_symmetric':
        max_abs = np.max(np.abs(data))
        return np.zeros_like(data) if max_abs < 1e-9 else data / max_abs
    return data

def sliding_window_view(arr, window_shape, step_size):
    window_h, window_w = window_shape if isinstance(window_shape, tuple) else (window_shape, window_shape)
    step_y, step_x = step_size if isinstance(step_size, tuple) else (step_size, step_size)
    arr_h, arr_w = arr.shape
    out_h = (arr_h - window_h) // step_y + 1
    out_w = (arr_w - window_w) // step_x + 1
    new_shape = (out_h, out_w, window_h, window_w)
    stride_y, stride_x = arr.strides
    new_strides = (stride_y * step_y, stride_x * step_x, stride_y, stride_x)
    return np.lib.stride_tricks.as_strided(arr, shape=new_shape, strides=new_strides)

def reconstruct_from_patches_average(patches, output_shape, window_size, step_size):
    output_h, output_w = output_shape
    reconstructed = np.zeros(output_shape, dtype=np.float32)
    count_map = np.zeros(output_shape, dtype=np.float32)
    patches = np.squeeze(patches)
    num_y = (output_h - window_size) // step_size + 1
    num_x = (output_w - window_size) // step_size + 1
    patch_idx = 0
    for yi in range(num_y):
        for xi in range(num_x):
            if patch_idx >= len(patches):
                break
            y0, y1 = yi * step_size, yi * step_size + window_size
            x0, x1 = xi * step_size, xi * step_size + window_size
            reconstructed[y0:y1, x0:x1] += patches[patch_idx]
            count_map[y0:y1, x0:x1] += 1.0
            patch_idx += 1
        if patch_idx >= len(patches):
            break
    count_map[count_map == 0] = 1.0
    return reconstructed / count_map

def calculate_nrms(data1, data2):
    rms_diff = np.sqrt(np.mean((data1 - data2) ** 2))
    denom = np.sqrt(np.mean(data1 ** 2)) + np.sqrt(np.mean(data2 ** 2))
    return 0.0 if denom == 0 else 200.0 * rms_diff / denom

# --- 4. Physics & Simulation Pipelines ---
def run_deepwave_simulation(vp_model_data, n_shots, n_receivers_per_shot, max_offset_m, dx, dt,
                             freq, nt, peak_time, source_depth, receiver_depth,
                             first_source, first_receiver, model_name, plot=False):
    vp_model_tensor = torch.from_numpy(vp_model_data).to(device, dtype=torch.float32)
    ny, nx_ = vp_model_tensor.shape

    source_locations = torch.zeros(n_shots, 1, 2, dtype=torch.long, device=device)
    receiver_locations = torch.zeros(n_shots, n_receivers_per_shot, 2, dtype=torch.long, device=device)

    max_offset_grid = int(max_offset_m / dx)
    regular_locations = torch.arange(n_shots, device=device, dtype=torch.float32) * (nx_ / n_shots) + first_source
    random_offsets = torch.randint(-max_offset_grid, max_offset_grid + 1, (n_shots,), device=device, dtype=torch.float32)
    randomized_x_locations = regular_locations + random_offsets
    randomized_x_locations.clamp_(0, nx_ - 1)
    source_locations[..., 1] = source_depth
    source_locations[:, 0, 0] = randomized_x_locations.long()
    print(f"[{model_name}] Randomized source locations, max spacing +/- {max_offset_m} m.")

    receiver_locations[..., 1] = receiver_depth
    receiver_locations[:, :, 0] = (torch.arange(n_receivers_per_shot) * (nx_ / n_receivers_per_shot) + first_receiver).repeat(n_shots, 1)

    source_amplitudes = (
        deepwave.wavelets.ricker(freq, nt, dt, peak_time).repeat(n_shots, 1, 1).to(device)
    )

    out_sim = scalar(
        vp_model_tensor.T, dx, dt,
        source_amplitudes=source_amplitudes,
        source_locations=source_locations,
        receiver_locations=receiver_locations,
        accuracy=4, pml_freq=freq
    )
    receiver_amplitudes = out_sim[-1]
    print(f"[{model_name}] Simulation complete, amplitudes shape {receiver_amplitudes.shape}.")
    return receiver_amplitudes, source_locations

def process_and_stack_dataset(seismic_data, dataset_name, source_locations_for_this_data,
                               save_dir_figures, save_dir_numpy, plot=False):
    print(f"Processing dataset: {dataset_name}")
    num_shots, num_receivers, num_samples = seismic_data.shape
    source_positions = source_locations_for_this_data[:, 0, 0].float() * dx
    receiver_spacing_m = d_receiver * dx
    receiver_positions = (torch.arange(num_receivers, device=device) * receiver_spacing_m) + (first_receiver_grid * dx)
    all_cmps = (source_positions.view(-1, 1) + receiver_positions.view(1, -1)) / 2
    all_offsets = torch.abs(source_positions.view(-1, 1) - receiver_positions.view(1, -1))

    fixed_cmp_min_m = 0.0
    fixed_cmp_max_m = nx * dx
    cmp_bin_size = receiver_spacing_m / 2
    global cmp_bins
    cmp_bins = torch.arange(fixed_cmp_min_m, fixed_cmp_max_m, cmp_bin_size, device=device)
    num_cmp_bins = len(cmp_bins)

    bin_indices = torch.bucketize(all_cmps, cmp_bins)
    cmp_gathers_list = [None] * num_cmp_bins
    unique_bins, _ = torch.unique(bin_indices, return_inverse=True)
    for bin_idx_val in unique_bins:
        bin_idx = bin_idx_val.item()
        if 0 <= bin_idx < num_cmp_bins:
            mask = (bin_indices == bin_idx_val)
            traces = seismic_data[mask]
            offsets = all_offsets[mask]
            sorted_idx = torch.argsort(offsets)
            cmp_gathers_list[bin_idx] = {'traces': traces[sorted_idx], 'offsets': offsets[sorted_idx]}

    def apply_nmo_correction(gather_data, dt, v_nmo_func, stretch_max=0.5):
        traces = gather_data['traces']
        offsets = gather_data['offsets']
        n_traces, n_samples = traces.shape
        if n_traces == 0:
            return None
        t0 = (torch.arange(n_samples, device=device, dtype=torch.float32) * dt).view(1, -1)
        x = offsets.view(-1, 1)
        if v_nmo_func.ndim == 1:
            v_nmo_func = v_nmo_func.view(1, -1)
        t_nmo = torch.sqrt(t0 ** 2 + (x / (v_nmo_func + 1e-9)) ** 2)

        # --- Calculate NMO Stretch ---
        stretch = (t_nmo - t0) / (t0 + 1e-6)
        # Create a smooth taper mask.
        stretch_taper_start = stretch_max - 0.15
        stretch_weights = 1.0 - torch.clamp((stretch - stretch_taper_start) / 0.15, min=0.0, max=1.0)

        sample_nmo = t_nmo / dt
        sample_floor = torch.floor(sample_nmo).long().clamp(0, n_samples - 2)
        sample_ceil = sample_floor + 1
        weight_ceil = sample_nmo - sample_floor.float()

        val_floor = torch.gather(traces, 1, sample_floor)
        val_ceil = torch.gather(traces, 1, sample_ceil)

        corrected_traces = val_floor * (1.0 - weight_ceil) + val_ceil * weight_ceil
        # Apply the stretch mute weight mask
        return corrected_traces * stretch_weights

    def apply_top_mute(corrected_gather, offsets, dt, slope=0.5, t0=0.1, taper_length_s=0.08):
        n_traces, n_samples = corrected_gather.shape
        mute_times = t0 + offsets * (slope / 1000)
        mute_samples = mute_times / dt
        time_samples = torch.arange(n_samples, device=device).expand_as(corrected_gather)

        # Calculate distance from the mute boundary
        dist_from_mute = time_samples - mute_samples.unsqueeze(1)
        # Create a smooth taper
        taper_samples = taper_length_s / dt
        weights = torch.clamp(dist_from_mute / taper_samples, min=0.0, max=1.0)

        return corrected_gather * weights

    nmo_velocities = torch.zeros(num_cmp_bins, num_samples, device=device)
    time_axis = torch.arange(num_samples, device=device) * dt
    vp_model_tensor = torch.from_numpy(baseline_data).to(device, dtype=torch.float32)  # NMO always uses baseline

    for i in range(num_cmp_bins):
        cmp_location_m = cmp_bins[i]
        x_idx = int(round((cmp_location_m / dx).item()))
        x_idx = min(x_idx, vp_model_tensor.shape[1] - 1)
        v_profile_depth = vp_model_tensor[:, x_idx]
        dz = dx
        twt_profile = torch.zeros_like(v_profile_depth, dtype=torch.float32)
        twt_profile[1:] = torch.cumsum(2 * dz / v_profile_depth[:-1], dim=0)
        v_nmo_t = torch.from_numpy(
            np.interp(time_axis.cpu().numpy(), twt_profile.cpu().numpy(), v_profile_depth.cpu().numpy())
        ).to(device)
        nmo_velocities[i, :] = v_nmo_t

    stacked_section = torch.zeros(num_cmp_bins, num_samples, device=device)
    for i, gather in enumerate(cmp_gathers_list):
        if gather is not None and len(gather['offsets']) > 0:
            corrected_traces = apply_nmo_correction(gather, dt, nmo_velocities[i])
            if corrected_traces is not None:
                muted_traces = apply_top_mute(corrected_traces, gather['offsets'], dt)
                stacked_section[i] = torch.mean(muted_traces, dim=0)

    stacked_np = stacked_section.cpu().numpy()
    stacked_smoothed = gaussian_filter1d(stacked_np, sigma=1.0, axis=0)

    os.makedirs(save_dir_numpy, exist_ok=True)
    np.save(os.path.join(save_dir_numpy, f'final_stacked_cmp_section_({dataset_name}).npy'), stacked_smoothed)
    print(f"  -> stacked section shape {stacked_smoothed.shape}, saved.")
    return stacked_smoothed

# --- 5. Data Loading & Generation ---
BASE_PATH = 'data/velocity_models/'

def load_bin_model(name):
    path = os.path.join(BASE_PATH, f'{name}.bin')
    data = np.fromfile(path, dtype=np.float32)
    return data.reshape((nx, nz)).T

baseline_data = load_bin_model('baseline')
monitoring_stage1_data = load_bin_model('monitoring_stage1')

print(f"baseline_data shape: {baseline_data.shape}, range [{baseline_data.min():.0f}, {baseline_data.max():.0f}] m/s")
print(f"monitoring_stage1_data shape: {monitoring_stage1_data.shape}, "
      f"range [{monitoring_stage1_data.min():.0f}, {monitoring_stage1_data.max():.0f}] m/s")

# Build shared Good Target
dense_data, dense_locs = run_deepwave_simulation(
    vp_model_data=monitoring_stage1_data, n_shots=N_SHOTS_GOOD, n_receivers_per_shot=300,
    max_offset_m=0.0, dx=dx, dt=dt, freq=freq, nt=nt, peak_time=peak_time,
    source_depth=source_depth_grid, receiver_depth=receiver_depth_grid,
    first_source=first_source_grid, first_receiver=first_receiver_grid,
    model_name='Good_Dense_Target')

stack_good = process_and_stack_dataset(
    seismic_data=dense_data, dataset_name="Good_Dense_Target",
    source_locations_for_this_data=dense_locs,
    save_dir_figures='outputs/datagen_ablation/figures', save_dir_numpy='outputs/datagen_ablation/stacked')

Y_full = normalize_data(stack_good.T)
print(f"\nShared Good target ready, shape {Y_full.shape}.")

# Build Bad Datasets
bad_stacks = {}
print("\n=== Pre-modeling scattering (proposed) ===")
scatter_data, scatter_locs = run_deepwave_simulation(
    vp_model_data=monitoring_stage1_data, n_shots=N_SHOTS_BAD, n_receivers_per_shot=300,
    max_offset_m=MAX_OFFSET_M, dx=dx, dt=dt, freq=freq, nt=nt, peak_time=peak_time,
    source_depth=source_depth_grid, receiver_depth=receiver_depth_grid,
    first_source=first_source_grid, first_receiver=first_receiver_grid,
    model_name='Bad_Scattering_Proposed')

stack_scatter = process_and_stack_dataset(
    seismic_data=scatter_data, dataset_name="Bad_Scattering_Proposed",
    source_locations_for_this_data=scatter_locs,
    save_dir_figures='outputs/datagen_ablation/figures', save_dir_numpy='outputs/datagen_ablation/stacked')

# Naive stretch masks
N_KEPT_BINS = 15

kept_bins_regular = np.linspace(0, Y_full.shape[1] - 1, N_KEPT_BINS).round().astype(int)
kept_bins_regular = np.unique(kept_bins_regular)
kept_bins_regular_m = cmp_bins[kept_bins_regular].cpu().numpy()

scatter_x_m = scatter_locs[:, 0, 0].float().cpu().numpy() * dx
kept_bins_scattered = np.array([torch.argmin(torch.abs(cmp_bins - x)).item() for x in scatter_x_m])
kept_bins_scattered = np.unique(kept_bins_scattered)
kept_bins_scattered_m = cmp_bins[kept_bins_scattered].cpu().numpy()

all_cols = np.arange(Y_full.shape[1])

naive_resampling_regular = Y_full.copy()
for row in range(Y_full.shape[0]):
    naive_resampling_regular[row, :] = np.interp(all_cols, kept_bins_regular, Y_full[row, kept_bins_regular])

naive_resampling_scattered = Y_full.copy()
for row in range(Y_full.shape[0]):
    naive_resampling_scattered[row, :] = np.interp(all_cols, kept_bins_scattered, Y_full[row, kept_bins_scattered])

# Plot Masking Data Comparison
fig = plt.figure(figsize=(24, 6))
gs = fig.add_gridspec(1, 5, width_ratios=[1, 1, 1, 0.15, 1])
extent = [0, nx * dx, nt * dt, 0]
vabs = np.percentile(np.abs(Y_full), 99)

test_panels = [
    ('Naive "stretch/guess"\n(regular positions)', naive_resampling_regular, kept_bins_regular_m, 'Kept CMP bins'),
    ('Naive "stretch/guess"\n(SCATTERED positions)', naive_resampling_scattered, kept_bins_scattered_m, 'Kept CMP bins'),
    ('Our method: "scattering"\n(real physics, real gaps)', normalize_data(stack_scatter.T), scatter_x_m, 'Active shots'),
]

axes = []
for i, (title, img, positions, label) in enumerate(test_panels):
    ax = fig.add_subplot(gs[0, i], sharey=axes[0] if axes else None)
    ax.imshow(img, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs, extent=extent)
    for x in positions:
        ax.axvline(x, color='lime', linestyle='--', linewidth=1, alpha=0.8)
    ax.scatter(positions, np.zeros_like(positions), marker='*', color='lime', s=120,
               edgecolor='black', zorder=5, label=label)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('CMP Position (m)', fontsize=12)
    ax.legend(loc='upper right', fontsize=8)
    axes.append(ax)
axes[0].set_ylabel('Two-Way Time (s)', fontsize=12)

ax_ref = fig.add_subplot(gs[0, 4], sharey=axes[0])
ax_ref.imshow(Y_full, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs, extent=extent)
ax_ref.set_title('REFERENCE\nReal Good target', fontsize=13, fontweight='bold')
ax_ref.set_xlabel('CMP Position (m)', fontsize=12)
for spine in ax_ref.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(2.5)

plt.suptitle("Curiosity test: naive stretch/guess (regular vs. scattered) vs. our method", fontsize=14)
plt.tight_layout()
plt.savefig('outputs/datagen_ablation/figures/naive_image_space_masking.png', dpi=300, bbox_inches='tight')
plt.close()

bad_stacks['naive_resampling_regular'] = naive_resampling_regular
bad_stacks['naive_resampling_scattered'] = naive_resampling_scattered
bad_stacks['scattering'] = normalize_data(stack_scatter.T)

# --- 6. U-Net Setup ---
random_flipper = tf.keras.layers.RandomFlip("horizontal")
random_rotator = tf.keras.layers.RandomRotation(factor=0.03)
random_translator = tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05)

def augment(x, y):
    images = tf.concat([x, y], axis=-1)
    images = random_flipper(images)
    images = random_rotator(images)
    images = random_translator(images)
    x, y = tf.split(images, num_or_size_splits=2, axis=-1)

    if tf.random.uniform(()) > 0.5:
        noise = tf.random.normal(shape=tf.shape(x), mean=0.0, stddev=0.05, dtype=x.dtype)
        x = x + noise
    return x, y

def build_unet(base_filters=32):
    f = base_filters
    inputs = layers.Input(shape=(WINDOW_SIZE, WINDOW_SIZE, 1))
    def block(x, n):
        x = layers.Conv2D(n, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(n, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        return x
    c1 = block(inputs, f);   p1 = layers.MaxPooling2D()(c1)
    c2 = block(p1, f*2);     p2 = layers.MaxPooling2D()(c2)
    c3 = block(p2, f*4);     p3 = layers.MaxPooling2D()(c3)
    c4 = block(p3, f*8);     p4 = layers.MaxPooling2D()(c4)
    c5 = block(p4, f*16)
    u6 = layers.Conv2D(f*8, 3, padding='same', activation='relu')(layers.UpSampling2D()(c5))
    c6 = block(layers.Concatenate()([u6, c4]), f*8)
    u7 = layers.Conv2D(f*4, 3, padding='same', activation='relu')(layers.UpSampling2D()(c6))
    c7 = block(layers.Concatenate()([u7, c3]), f*4)
    u8 = layers.Conv2D(f*2, 3, padding='same', activation='relu')(layers.UpSampling2D()(c7))
    c8 = block(layers.Concatenate()([u8, c2]), f*2)
    u9 = layers.Conv2D(f, 3, padding='same', activation='relu')(layers.UpSampling2D()(c8))
    c9 = block(layers.Concatenate()([u9, c1]), f)
    outputs = layers.Conv2D(1, 1, activation='linear')(c9)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4, clipvalue=1.0),
                  loss=tf.keras.losses.MeanAbsoluteError(), metrics=['mae'])
    return model

class IncrementalHistorySaver(Callback):
    def __init__(self, path):
        super().__init__()
        self.path = path
        self.history = {'loss': [], 'mae': [], 'val_loss': [], 'val_mae': []}
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        for k in self.history:
            if k in logs:
                self.history[k].append(float(logs[k]))
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        with open(self.path, 'w') as f:
            json.dump(self.history, f, indent=2)

# --- 7. Train 3 Models ---
trained_models = {}
test_sets = {}
histories = {}

for method_name, X_stack in bad_stacks.items():
    print(f"\n{'='*75}\nTraining model for method: {method_name}\n{'='*75}")

    X_view = sliding_window_view(X_stack, window_shape=(WINDOW_SIZE, WINDOW_SIZE), step_size=STEP_SIZE)
    Y_view = sliding_window_view(Y_full, window_shape=(WINDOW_SIZE, WINDOW_SIZE), step_size=STEP_SIZE)
    X_segments = X_view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE).copy()[..., np.newaxis]
    Y_segments = Y_view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE).copy()[..., np.newaxis]

    X_train, X_temp, Y_train, Y_temp = train_test_split(
        X_segments, Y_segments, test_size=TEST_SIZE_TEMP, random_state=RANDOM_STATE)
    X_val, X_test, Y_val, Y_test = train_test_split(
        X_temp, Y_temp, test_size=VALIDATION_SIZE_FINAL, random_state=RANDOM_STATE)

    test_sets[method_name] = (X_test, Y_test)

    model = build_unet(base_filters=32)
    history_path = f'outputs/datagen_ablation/training_logs/history_{method_name}.json'
    saver = IncrementalHistorySaver(history_path)
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
    checkpoint = ModelCheckpoint(f'outputs/datagen_ablation/model_{method_name}.keras',
                                  monitor='val_loss', save_best_only=True, verbose=0)

    train_ds = (tf.data.Dataset.from_tensor_slices((X_train, Y_train))
                .cache().shuffle(1000).batch(BATCH_SIZE)
                .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
                .prefetch(tf.data.AUTOTUNE))
    val_ds = (tf.data.Dataset.from_tensor_slices((X_val, Y_val))
              .batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE))

    t0 = time.time()
    try:
        model.fit(train_ds, validation_data=val_ds, epochs=150,
                  callbacks=[early_stop, checkpoint, saver], verbose=1)
    except KeyboardInterrupt:
        print(f"  Interrupted -- progress through this point is saved to {history_path};")
        raise

    trained_models[method_name] = model
    histories[method_name] = saver.history

print(f"\nAll three models trained: {list(trained_models.keys())}")

# --- 8. Cross-Test Evaluation ---
results = {}
for train_method, model in trained_models.items():
    for test_method, (X_test, Y_test) in test_sets.items():
        pred = model.predict(X_test, verbose=0)
        pred_flat = pred.reshape(pred.shape[0], -1)
        true_flat = Y_test.reshape(Y_test.shape[0], -1)
        nrms = calculate_nrms(pred_flat, true_flat)
        s = np.mean([ssim(Y_test[i,...,0], pred[i,...,0],
                           data_range=Y_test[i,...,0].max() - Y_test[i,...,0].min() + 1e-9)
                     for i in range(min(len(Y_test), 200))])
        results[(train_method, test_method)] = (nrms, s)

methods = list(bad_stacks.keys())
print(f"{'Trained on \ Tested on':28s}" + "".join(f"{m:>22s}" for m in methods))
for train_method in methods:
    row = f"{train_method:28s}"
    for test_method in methods:
        nrms, s = results[(train_method, test_method)]
        row += f"  NRMS={nrms:5.1f}% SSIM={s:.3f}"
    print(row)

with open('outputs/datagen_ablation/cross_test_results.json', 'w') as f:
    json.dump({f"{k[0]}_on_{k[1]}": v for k, v in results.items()}, f, indent=2)

# --- 9. Full-Image Reconstructions & Final Matrix Plot ---
full_predictions = {}
full_results = {}

for train_method, model in trained_models.items():
    for test_method, test_stack in bad_stacks.items():
        view = sliding_window_view(test_stack, window_shape=(WINDOW_SIZE, WINDOW_SIZE), step_size=STEP_SIZE)
        patches = view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE).copy()[..., np.newaxis]
        pred_patches = model.predict(patches, verbose=0)
        pred_image = reconstruct_from_patches_average(pred_patches, test_stack.shape, WINDOW_SIZE, STEP_SIZE)

        full_predictions[(train_method, test_method)] = pred_image
        nrms = calculate_nrms(pred_image, Y_full)
        s = ssim(Y_full, pred_image, data_range=Y_full.max() - Y_full.min())
        full_results[(train_method, test_method)] = (nrms, s)

n = len(methods)
fig, axes = plt.subplots(n, n, figsize=(4*n, 4*n), sharex=True, sharey=True)
for i, train_method in enumerate(methods):
    for j, test_method in enumerate(methods):
        ax = axes[i, j] if n > 1 else axes
        img = full_predictions[(train_method, test_method)]
        nrms, s = full_results[(train_method, test_method)]
        ax.imshow(img, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs, extent=extent)
        ax.set_title(f'NRMS={nrms:.1f}% SSIM={s:.3f}', fontsize=9)
        if i == 0:
            ax.annotate(f'Tested on:\n{test_method}', xy=(0.5, 1.25), xycoords='axes fraction',
                        ha='center', fontsize=10, fontweight='bold')
        if j == 0:
            ax.set_ylabel(f'Trained on:\n{train_method}', fontsize=10, fontweight='bold')

plt.suptitle('Predicted sections: every trained model x every test method', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('outputs/datagen_ablation/figures/prediction_grid.png', dpi=300, bbox_inches='tight')
plt.close()

nrms_matrix = np.array([[full_results[(tr, te)][0] for te in methods] for tr in methods])
ssim_matrix = np.array([[full_results[(tr, te)][1] for te in methods] for tr in methods])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7*n/3+3, 6))

im1 = ax1.imshow(nrms_matrix, cmap='RdYlGn_r', vmin=0, vmax=max(100, nrms_matrix.max()))
ax1.set_xticks(range(n)); ax1.set_xticklabels(methods, rotation=45, ha='right')
ax1.set_yticks(range(n)); ax1.set_yticklabels(methods)
ax1.set_xlabel('Tested on'); ax1.set_ylabel('Trained on')
ax1.set_title('NRMS (%) -- lower is better')
for i in range(n):
    for j in range(n):
        ax1.text(j, i, f'{nrms_matrix[i,j]:.1f}', ha='center', va='center', fontsize=10)
plt.colorbar(im1, ax=ax1, fraction=0.046)

im2 = ax2.imshow(ssim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax2.set_xticks(range(n)); ax2.set_xticklabels(methods, rotation=45, ha='right')
ax2.set_yticks(range(n)); ax2.set_yticklabels(methods)
ax2.set_xlabel('Tested on'); ax2.set_ylabel('Trained on')
ax2.set_title('SSIM -- higher is better')
for i in range(n):
    for j in range(n):
        ax2.text(j, i, f'{ssim_matrix[i,j]:.3f}', ha='center', va='center', fontsize=10)
plt.colorbar(im2, ax=ax2, fraction=0.046)

plt.suptitle('Cross-test results matrix (whole-image evaluation)', fontsize=15)
plt.tight_layout()
plt.savefig('outputs/datagen_ablation/figures/results_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

print("\nProcess Complete! Ablation matrices and results saved to outputs/datagen_ablation/ directory.")

## 2. Cross-test and figures

In [ ]:
# ==============================================================================
# Visualization Cell: Three-Way Data-Generation Ablation Study
#
# Loads pre-trained models and saved numpy datasets from outputs/datagen_ablation/
# to generate the input comparison, prediction grid, and metric matrices.
# ==============================================================================

import os
import json
import numpy as np
import torch
import tensorflow as tf
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

print("--- Initializing Plotting Environment ---")

# --- 1. Parameters & Setup ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

dx = 2
dt = 0.004
nt = 300
nx = 601
d_receiver = nx / 300
first_receiver_grid = 0
first_source = 5

WINDOW_SIZE = 128
STEP_SIZE = 5

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

# --- 2. Helper Functions ---
def normalize_data(data, method='minmax_symmetric'):
    if method == 'minmax_symmetric':
        max_abs = np.max(np.abs(data))
        return np.zeros_like(data) if max_abs < 1e-9 else data / max_abs
    return data

def sliding_window_view(arr, window_shape, step_size):
    window_h, window_w = window_shape if isinstance(window_shape, tuple) else (window_shape, window_shape)
    step_y, step_x = step_size if isinstance(step_size, tuple) else (step_size, step_size)
    arr_h, arr_w = arr.shape
    out_h = (arr_h - window_h) // step_y + 1
    out_w = (arr_w - window_w) // step_x + 1
    new_shape = (out_h, out_w, window_h, window_w)
    stride_y, stride_x = arr.strides
    new_strides = (stride_y * step_y, stride_x * step_x, stride_y, stride_x)
    return np.lib.stride_tricks.as_strided(arr, shape=new_shape, strides=new_strides)

def reconstruct_from_patches_average(patches, output_shape, window_size, step_size):
    output_h, output_w = output_shape
    reconstructed = np.zeros(output_shape, dtype=np.float32)
    count_map = np.zeros(output_shape, dtype=np.float32)
    patches = np.squeeze(patches)
    num_y = (output_h - window_size) // step_size + 1
    num_x = (output_w - window_size) // step_size + 1
    patch_idx = 0
    for yi in range(num_y):
        for xi in range(num_x):
            if patch_idx >= len(patches): break
            y0, y1 = yi * step_size, yi * step_size + window_size
            x0, x1 = xi * step_size, xi * step_size + window_size
            reconstructed[y0:y1, x0:x1] += patches[patch_idx]
            count_map[y0:y1, x0:x1] += 1.0
            patch_idx += 1
    count_map[count_map == 0] = 1.0
    return reconstructed / count_map

def calculate_nrms(data1, data2):
    rms_diff = np.sqrt(np.mean((data1 - data2) ** 2))
    denom = np.sqrt(np.mean(data1 ** 2)) + np.sqrt(np.mean(data2 ** 2))
    return 0.0 if denom == 0 else 200.0 * rms_diff / denom

# --- 3. Recreate Masking Logic & Load Data ---
print("Loading saved baseline datasets...")
try:
    stack_good = np.load('outputs/datagen_ablation/stacked/final_stacked_cmp_section_(Good_Dense_Target).npy')
    stack_scatter = np.load('outputs/datagen_ablation/stacked/final_stacked_cmp_section_(Bad_Scattering_Proposed).npy')
except FileNotFoundError:
    raise FileNotFoundError("Could not find the stacked .npy files. Make sure 'outputs/datagen_ablation/stacked/' exists and contains the files.")

Y_full = normalize_data(stack_good.T)
scattering_data = normalize_data(stack_scatter.T)

# Re-calculate cmp bins
receiver_spacing_m = d_receiver * dx
fixed_cmp_max_m = nx * dx
cmp_bin_size = receiver_spacing_m / 2
cmp_bins = np.arange(0.0, fixed_cmp_max_m, cmp_bin_size)

# Recreate Random Scatter Locations (Matches Training RNG perfectly)
n_shots_bad = 15
max_offset_m = 150.0
max_offset_grid = int(max_offset_m / dx)

regular_locations = torch.arange(n_shots_bad, dtype=torch.float32) * (nx / n_shots_bad) + first_source
random_offsets = torch.randint(-max_offset_grid, max_offset_grid + 1, (n_shots_bad,), dtype=torch.float32)
randomized_x_locations = regular_locations + random_offsets
randomized_x_locations.clamp_(0, nx - 1)
scatter_x_m = randomized_x_locations.numpy() * dx

# Determine Kept Bins
N_KEPT_BINS = 15
kept_bins_regular = np.linspace(0, Y_full.shape[1] - 1, N_KEPT_BINS).round().astype(int)
kept_bins_regular = np.unique(kept_bins_regular)
kept_bins_regular_m = cmp_bins[kept_bins_regular]

kept_bins_scattered = np.array([np.argmin(np.abs(cmp_bins - x)) for x in scatter_x_m])
kept_bins_scattered = np.unique(kept_bins_scattered)
kept_bins_scattered_m = cmp_bins[kept_bins_scattered]

# Create Naive Decimation Arrays
all_cols = np.arange(Y_full.shape[1])
naive_resampling_regular = Y_full.copy()
for row in range(Y_full.shape[0]):
    naive_resampling_regular[row, :] = np.interp(all_cols, kept_bins_regular, Y_full[row, kept_bins_regular])

naive_resampling_scattered = Y_full.copy()
for row in range(Y_full.shape[0]):
    naive_resampling_scattered[row, :] = np.interp(all_cols, kept_bins_scattered, Y_full[row, kept_bins_scattered])

bad_stacks = {
    'naive_resampling_regular': naive_resampling_regular,
    'naive_resampling_scattered': naive_resampling_scattered,
    'scattering': scattering_data
}
methods = list(bad_stacks.keys())

# --- 4. Plotting Fig 1: Input Data Comparison ---
print("Generating Input Data Comparison Plot...")
os.makedirs('outputs/datagen_ablation/figures', exist_ok=True)

fig = plt.figure(figsize=(24, 6))
gs = fig.add_gridspec(1, 5, width_ratios=[1, 1, 1, 0.15, 1])
extent = [0, nx * dx, nt * dt, 0]
vabs = np.percentile(np.abs(Y_full), 99)

test_panels = [
    ('Naive "stretch/guess"\n(regular positions)', naive_resampling_regular, kept_bins_regular_m, 'Kept CMP bins'),
    ('Naive "stretch/guess"\n(SCATTERED positions)', naive_resampling_scattered, kept_bins_scattered_m, 'Kept CMP bins'),
    ('Our method: "scattering"\n(real physics, real gaps)', scattering_data, scatter_x_m, 'Active shots'),
]

axes = []
for i, (title, img, positions, label) in enumerate(test_panels):
    ax = fig.add_subplot(gs[0, i], sharey=axes[0] if axes else None)
    ax.imshow(img, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs, extent=extent)
    for x in positions:
        ax.axvline(x, color='lime', linestyle='--', linewidth=1, alpha=0.8)
    ax.scatter(positions, np.zeros_like(positions), marker='*', color='lime', s=120,
               edgecolor='black', zorder=5, label=label)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('CMP Position (m)', fontsize=12)
    ax.legend(loc='upper right', fontsize=8)
    axes.append(ax)
axes[0].set_ylabel('Two-Way Time (s)', fontsize=12)

ax_ref = fig.add_subplot(gs[0, 4], sharey=axes[0])
ax_ref.imshow(Y_full, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs, extent=extent)
ax_ref.set_title('REFERENCE\nReal Good target', fontsize=13, fontweight='bold')
ax_ref.set_xlabel('CMP Position (m)', fontsize=12)
for spine in ax_ref.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(2.5)

plt.suptitle("Curiosity test: naive stretch/guess (regular vs. scattered) vs. our method", fontsize=14)
plt.tight_layout()
plt.savefig('outputs/datagen_ablation/figures/naive_image_space_masking_replot.png', dpi=300, bbox_inches='tight')
plt.show()

# --- 5. Loading Models & Running Inference ---
full_predictions = {}
full_results = {}

print("\nLoading models and predicting on full images...")
for train_method in methods:
    model_path = f'outputs/datagen_ablation/model_{train_method}.keras'
    if not os.path.exists(model_path):
        print(f"⚠️ Warning: Model {model_path} not found. Skipping.")
        continue

    model = tf.keras.models.load_model(model_path, compile=False)

    for test_method, test_stack in bad_stacks.items():
        view = sliding_window_view(test_stack, window_shape=(WINDOW_SIZE, WINDOW_SIZE), step_size=STEP_SIZE)
        patches = view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE).copy()[..., np.newaxis]

        pred_patches = model.predict(patches, batch_size=32, verbose=0)
        pred_image = reconstruct_from_patches_average(pred_patches, test_stack.shape, WINDOW_SIZE, STEP_SIZE)

        # Remove DC Bias for perfect visualization
        pred_image = pred_image - np.mean(pred_image)

        full_predictions[(train_method, test_method)] = pred_image
        nrms = calculate_nrms(pred_image, Y_full)
        s = ssim(Y_full, pred_image, data_range=Y_full.max() - Y_full.min())
        full_results[(train_method, test_method)] = (nrms, s)

# Corrected string formatting here
print(f"\n{'Trained on / Tested on':28s}" + "".join(f"{m:>28s}" for m in methods))
for train_method in methods:
    row = f"{train_method:28s}"
    for test_method in methods:
        nrms, s = full_results[(train_method, test_method)]
        row += f"  NRMS={nrms:5.1f}% SSIM={s:.3f}"
    print(row)

# --- 6. Plotting Fig 2: 3x3 Prediction Grid ---
print("\nGenerating Prediction Grid Plot...")
n = len(methods)
fig, axes = plt.subplots(n, n, figsize=(4*n, 4*n), sharex=True, sharey=True)
for i, train_method in enumerate(methods):
    for j, test_method in enumerate(methods):
        ax = axes[i, j] if n > 1 else axes
        img = full_predictions.get((train_method, test_method), np.zeros_like(Y_full))
        nrms, s = full_results.get((train_method, test_method), (100.0, 0.0))

        ax.imshow(img, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs, extent=extent)
        ax.set_title(f'NRMS={nrms:.1f}% SSIM={s:.3f}', fontsize=10)

        if i == 0:
            ax.annotate(f'Tested on:\n{test_method}', xy=(0.5, 1.25), xycoords='axes fraction',
                        ha='center', fontsize=11, fontweight='bold')
        if j == 0:
            ax.set_ylabel(f'Trained on:\n{train_method}', fontsize=11, fontweight='bold')

plt.suptitle('Predicted sections: every trained model vs. every test method', fontsize=15, y=1.04)
plt.tight_layout()
plt.savefig('outputs/datagen_ablation/figures/prediction_grid_replot.png', dpi=300, bbox_inches='tight')
plt.show()

# --- 7. Plotting Fig 3: Metric Matrices ---
print("\nGenerating Performance Matrix Plot...")
nrms_matrix = np.array([[full_results[(tr, te)][0] for te in methods] for tr in methods])
ssim_matrix = np.array([[full_results[(tr, te)][1] for te in methods] for tr in methods])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7*n/3+3, 6))

im1 = ax1.imshow(nrms_matrix, cmap='RdYlGn_r', vmin=0, vmax=max(100, nrms_matrix.max()))
ax1.set_xticks(range(n)); ax1.set_xticklabels(methods, rotation=45, ha='right')
ax1.set_yticks(range(n)); ax1.set_yticklabels(methods)
ax1.set_xlabel('Tested on', fontweight='bold'); ax1.set_ylabel('Trained on', fontweight='bold')
ax1.set_title('NRMS (%) -- lower is better', fontsize=12)
for i in range(n):
    for j in range(n):
        ax1.text(j, i, f'{nrms_matrix[i,j]:.1f}', ha='center', va='center', fontsize=12,
                 color="white" if nrms_matrix[i,j] > np.median(nrms_matrix) or nrms_matrix[i,j] < 15 else "black")
plt.colorbar(im1, ax=ax1, fraction=0.046)

im2 = ax2.imshow(ssim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax2.set_xticks(range(n)); ax2.set_xticklabels(methods, rotation=45, ha='right')
ax2.set_yticks(range(n)); ax2.set_yticklabels(methods)
ax2.set_xlabel('Tested on', fontweight='bold'); ax2.set_ylabel('Trained on', fontweight='bold')
ax2.set_title('SSIM -- higher is better', fontsize=12)
for i in range(n):
    for j in range(n):
        ax2.text(j, i, f'{ssim_matrix[i,j]:.3f}', ha='center', va='center', fontsize=12,
                 color="white" if ssim_matrix[i,j] < np.median(ssim_matrix) or ssim_matrix[i,j] > 0.9 else "black")
plt.colorbar(im2, ax=ax2, fraction=0.046)

plt.suptitle('Cross-test results matrix (whole-image evaluation)', fontsize=15)
plt.tight_layout()
plt.savefig('outputs/datagen_ablation/figures/results_matrix_replot.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Process Complete! All plots successfully generated and saved.")

In [ ]:
fig, axes = plt.subplots(n, n, figsize=(12, 11), sharex=True, sharey=True)
for i, train_method in enumerate(methods):
    for j, test_method in enumerate(methods):
        ax = axes[i, j]
        img = full_predictions[(train_method, test_method)]
        nrms, s = full_results[(train_method, test_method)]
        ax.imshow(img, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs, extent=extent)
        ax.set_title(f'NRMS = {nrms:.1f}%,  SSIM = {s:.3f}', fontsize=10)
        if i == 0:
            ax.annotate(f'Tested on: {short[j]}', xy=(0.5, 1.22), xycoords='axes fraction',
                        ha='center', fontsize=12, fontweight='bold')
        if j == 0:
            ax.set_ylabel(f'Trained on:\n{short[i]}\n\nTwo-way time (s)', fontsize=10, fontweight='bold')
        if i == n - 1:
            ax.set_xlabel('CMP position (m)', fontsize=10)

fig.tight_layout()
fig.savefig('outputs/datagen_ablation/figures/prediction_grid.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()